# prm-rl — Reward-Hacking Quickstart (Colab)

This notebook is the *smallest working demo* of the framework. It walks
through the pipeline end-to-end with a tiny model so it fits on Colab's
free T4 GPU (or CPU in a pinch), then trains **Arm 1 (Outcome-based)**
and **Arm 2 (Naive Process Reward)** and evaluates both.

Pipeline:

1. Install dependencies + clone the repo.
2. Load GSM8K and build a tiny golden subset (100 examples).
3. Supervised fine-tune a **SmolLM2-135M** policy on the golden subset.
4. Train a small **DeBERTa** Process Reward Model on step-level labels.
5. Run RL with **Arm 1** — outcome reward — using `trl.GRPOTrainer`.
6. Run RL with **Arm 2** — PRM step-sum reward — using the same trainer,
   just swapping the `reward_funcs` list.
7. Evaluate both policies: accuracy, exploit rate on trap prompts,
   verbosity, and a rough composite reward-hacking score.

> **Note** — every "experimental arm" from the research plan reduces to
> a different `reward_funcs` list in `trl.GRPOTrainer` — there is **no
> custom PPO/GRPO loop** anywhere in this codebase. This is by design.

## 0. Setup

**Colab**: just run the next cell. It clones the repo, `cd`s into it, and installs
`prm_rl` as an editable package so `import prm_rl` works.

**Local**: open this notebook from `prm_rl/notebooks/` (or `cd` into `prm_rl/`
before running). The same cell auto-detects local mode.

If you're using a fork or different branch, edit `REPO_URL` / `BRANCH` below.

In [ ]:
# ---------------- edit for your fork if needed ----------------
REPO_URL = "https://github.com/rishuray123/prm-rl.git"
BRANCH   = "main"
REPO_DIR = "prm-rl"
# --------------------------------------------------------------

import os, sys, pathlib, subprocess

IN_COLAB = "google.colab" in sys.modules

if IN_COLAB:
    if not pathlib.Path(REPO_DIR).exists():
        subprocess.check_call(["git", "clone", "-b", BRANCH, REPO_URL, REPO_DIR])
    ROOT = pathlib.Path(REPO_DIR).resolve()
else:
    # Local: run from prm_rl/ or from prm_rl/notebooks/
    cwd = pathlib.Path.cwd()
    ROOT = cwd.parent if cwd.name == "notebooks" else cwd

os.chdir(ROOT)
print("Repo root :", ROOT)
print("In Colab  :", IN_COLAB)
assert (ROOT / "pyproject.toml").exists(), \
    f"pyproject.toml not found at {ROOT}; edit REPO_URL or start from prm_rl/"

# Make `import prm_rl` work immediately, without relying on pip having
# refreshed site-packages for this already-running kernel. The next cell
# still runs `pip install -e .` for dependency resolution + console
# scripts, but this line guarantees imports work even before that.
SRC = str((ROOT / "src").resolve())
if SRC not in sys.path:
    sys.path.insert(0, SRC)
print("sys.path[0]:", sys.path[0])

In [ ]:
# Install `prm_rl` in editable mode. pyproject.toml lists all runtime
# deps (transformers, trl, datasets, accelerate, peft, omegaconf,
# rouge-score, scikit-learn, sentencepiece). Colab already ships torch,
# so this is fast. We *don't* pass -q so any dep-resolver conflict is
# visible in-notebook.
subprocess.check_call([sys.executable, "-m", "pip", "install", "-e", "."])

# Refresh site so any new .pth files (editable install) are picked up
# by the already-running kernel. Belt-and-suspenders with the sys.path
# hack in the previous cell.
import importlib, site
importlib.reload(site)

# Smoke-test: this must succeed or the rest of the notebook won't run.
import prm_rl
print("prm_rl imported from:", prm_rl.__file__)

In [ ]:
import torch, transformers, trl, datasets
print('torch     :', torch.__version__, '| CUDA:', torch.cuda.is_available(),
      '|', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU')
print('transformers:', transformers.__version__)
print('trl         :', trl.__version__)
print('datasets    :', datasets.__version__)

## 1. Load GSM8K & build a tiny golden subset

For the quickstart we cap everything at **100 training problems**. The
goal is to see the wiring work end-to-end; scaling up is just changing
`N_TRAIN` and moving off Colab.

In [ ]:
from prm_rl.data.gsm8k import load_gsm8k
from prm_rl.data.golden import build_golden_dataset
from prm_rl.data.prm_data import build_prm_dataset

N_TRAIN = 100
N_TEST  = 50

gsm_train = load_gsm8k(split='train', n=N_TRAIN, seed=0)
gsm_test  = load_gsm8k(split='test',  n=N_TEST,  seed=0)
print('train:', len(gsm_train), 'test:', len(gsm_test))
print(gsm_train[0])

In [ ]:
golden = build_golden_dataset(gsm_train, strategy='gsm8k_native')
print('golden columns:', golden.column_names)
print('example steps:', golden[0]['steps'][:3], '...')
print('example labels:', golden[0]['step_labels'])

In [ ]:
prm_ds = build_prm_dataset(golden)
print('PRM rows:', len(prm_ds))
print(prm_ds[0])

## 2. Supervised fine-tune a tiny policy on the golden set

Uses `trl.SFTTrainer`. We deliberately pick a very small model
(`SmolLM2-135M-Instruct`, ~135 M parameters) so an epoch takes seconds
on a T4. Swap `MODEL` to `Qwen/Qwen2.5-0.5B-Instruct` for a slightly
stronger baseline.

In [ ]:
MODEL = 'HuggingFaceTB/SmolLM2-135M-Instruct'
OUT_SFT = 'outputs/sft'

from transformers import AutoModelForCausalLM, AutoTokenizer
from trl import SFTConfig, SFTTrainer

tokenizer = AutoTokenizer.from_pretrained(MODEL)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = 'left'

model = AutoModelForCausalLM.from_pretrained(
    MODEL,
    torch_dtype=torch.bfloat16 if torch.cuda.is_available() else torch.float32,
    device_map='auto',
)

def _fmt(ex):
    return {'text': f"{ex['prompt']}\n\n{ex['trace']}"}
sft_train = golden.map(_fmt, remove_columns=golden.column_names)
print(sft_train[0]['text'][:400], '...')

In [ ]:
sft_cfg = SFTConfig(
    output_dir=OUT_SFT,
    num_train_epochs=1,
    per_device_train_batch_size=4,
    gradient_accumulation_steps=1,
    learning_rate=2e-5,
    logging_steps=5,
    save_strategy='no',
    bf16=torch.cuda.is_available(),
    max_seq_length=768,
    packing=False,
    report_to='none',
    dataset_text_field='text',
)
sft = SFTTrainer(model=model, tokenizer=tokenizer, args=sft_cfg, train_dataset=sft_train)
sft.train()
sft.save_model(OUT_SFT); tokenizer.save_pretrained(OUT_SFT)

## 3. Train the Process Reward Model (Arm 2 uses this)

A stock 2-class DeBERTa classifier over `question + steps[:i]`. Because
our tiny native-GSM8K golden set has label=1 for every step, this is a
cheap sanity training run — in the full framework you'd use
`strategy='verifier'` when building the golden set so the PRM sees
genuine positives and negatives.

To keep this section runnable on free Colab we cap epochs to 1.

In [ ]:
PRM_MODEL = 'microsoft/deberta-v3-xsmall'   # tiny, ~22M params
OUT_PRM   = 'outputs/prm'

import numpy as np
from transformers import (
    AutoModelForSequenceClassification, AutoTokenizer,
    Trainer, TrainingArguments, DataCollatorWithPadding,
)
from sklearn.metrics import accuracy_score, f1_score

# Sprinkle in some synthetic negatives so the classifier has both labels.
# For real experiments, use strategy='verifier' when building `golden`.
import random, copy
rng = random.Random(0)
prm_rows = [dict(r) for r in prm_ds]
for row in list(prm_rows):
    if rng.random() < 0.3:
        neg = copy.deepcopy(row)
        neg['text'] += '\n\nTherefore the answer is 999999.'
        neg['label'] = 0
        prm_rows.append(neg)
rng.shuffle(prm_rows)
from datasets import Dataset
prm_ds2 = Dataset.from_list(prm_rows)
print('PRM rows after augmentation:', len(prm_ds2),
      '| positives:', sum(r['label'] for r in prm_rows))

In [ ]:
prm_tok = AutoTokenizer.from_pretrained(PRM_MODEL)
prm_model = AutoModelForSequenceClassification.from_pretrained(PRM_MODEL, num_labels=2)

def tok_fn(batch):
    enc = prm_tok(batch['text'], truncation=True, max_length=512, padding=False)
    enc['labels'] = batch['label']
    return enc

split = prm_ds2.train_test_split(test_size=0.1, seed=0)
train_prm = split['train'].map(tok_fn, batched=True, remove_columns=split['train'].column_names)
val_prm   = split['test' ].map(tok_fn, batched=True, remove_columns=split['test' ].column_names)

def _metrics(p):
    preds = np.argmax(p.predictions, axis=-1)
    return {'accuracy': accuracy_score(p.label_ids, preds),
            'f1': f1_score(p.label_ids, preds, zero_division=0)}

prm_args = TrainingArguments(
    output_dir=OUT_PRM,
    num_train_epochs=1,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=32,
    learning_rate=2e-5,
    eval_strategy='epoch',
    save_strategy='no',
    logging_steps=10,
    bf16=torch.cuda.is_available(),
    report_to='none',
)
prm_trainer = Trainer(
    model=prm_model, args=prm_args,
    train_dataset=train_prm, eval_dataset=val_prm,
    tokenizer=prm_tok, data_collator=DataCollatorWithPadding(prm_tok),
    compute_metrics=_metrics,
)
prm_trainer.train()
prm_trainer.save_model(OUT_PRM); prm_tok.save_pretrained(OUT_PRM)

## 4. RL — **Arm 1: Outcome-based reward**

The reward function is a plain Python callable that returns one float
per completion. GRPO takes a *list* of these; Arm 1 uses just one.

In [ ]:
from prm_rl.rewards.outcome  import outcome_reward
from prm_rl.rewards.process  import make_naive_process_reward
from prm_rl.models.prm       import load_prm
from prm_rl.utils.steps      import extract_final_answer, split_steps

# Quick sanity check of the reward on a fake batch.
prompts     = [gsm_test[0]['prompt']]
completions = ['Some reasoning...\n\n#### ' + gsm_test[0]['answer']]
print('Arm-1 reward on a correct fake completion:',
      outcome_reward(prompts, completions, answer=[gsm_test[0]['answer']]))

In [ ]:
OUT_ARM1 = 'outputs/arm1_outcome'

from trl import GRPOConfig, GRPOTrainer
from transformers import AutoModelForCausalLM, AutoTokenizer

arm1_model = AutoModelForCausalLM.from_pretrained(
    OUT_SFT,
    torch_dtype=torch.bfloat16 if torch.cuda.is_available() else torch.float32,
    device_map='auto',
)
arm1_tok   = AutoTokenizer.from_pretrained(OUT_SFT)
if arm1_tok.pad_token is None:
    arm1_tok.pad_token = arm1_tok.eos_token

# GRPO wants the training dataset to expose the prompt as `prompt`.
rl_train = gsm_train.remove_columns([c for c in gsm_train.column_names if c not in {'prompt','question','answer'}])

arm1_cfg = GRPOConfig(
    output_dir=OUT_ARM1,
    max_steps=20,                      # tiny for Colab
    per_device_train_batch_size=2,
    gradient_accumulation_steps=1,
    learning_rate=5e-6,
    logging_steps=5,
    save_strategy='no',
    bf16=torch.cuda.is_available(),
    num_generations=2,
    max_prompt_length=384,
    max_completion_length=256,
    temperature=0.9,
    beta=0.04,
    report_to='none',
    reward_weights=[1.0],
)
arm1_trainer = GRPOTrainer(
    model=arm1_model,
    processing_class=arm1_tok,
    args=arm1_cfg,
    train_dataset=rl_train,
    reward_funcs=[outcome_reward],
)
arm1_trainer.train()
arm1_trainer.save_model(OUT_ARM1); arm1_tok.save_pretrained(OUT_ARM1)

## 5. RL — **Arm 2: Naive Process Reward**

Same trainer, same dataset, same hyperparameters. The **only** thing
that differs from Arm 1 is `reward_funcs` and `reward_weights`. This is
the whole point of the framework.

In [ ]:
OUT_ARM2 = 'outputs/arm2_naive_process'

prm_scorer = load_prm(OUT_PRM, device='cuda' if torch.cuda.is_available() else 'cpu')
process_reward = make_naive_process_reward(prm=prm_scorer, aggregation='mean')

# Sanity check on a well-formed fake completion.
print('Arm-2 reward on a fake completion:',
      process_reward([gsm_test[0]['prompt']],
                     ['We compute step 1.\n\nWe compute step 2.\n\n#### 42'],
                     question=[gsm_test[0]['question']]))

In [ ]:
arm2_model = AutoModelForCausalLM.from_pretrained(
    OUT_SFT,
    torch_dtype=torch.bfloat16 if torch.cuda.is_available() else torch.float32,
    device_map='auto',
)
arm2_tok = AutoTokenizer.from_pretrained(OUT_SFT)
if arm2_tok.pad_token is None:
    arm2_tok.pad_token = arm2_tok.eos_token

arm2_cfg = GRPOConfig(
    output_dir=OUT_ARM2,
    max_steps=20,
    per_device_train_batch_size=2,
    gradient_accumulation_steps=1,
    learning_rate=5e-6,
    logging_steps=5,
    save_strategy='no',
    bf16=torch.cuda.is_available(),
    num_generations=2,
    max_prompt_length=384,
    max_completion_length=256,
    temperature=0.9,
    beta=0.04,
    report_to='none',
    reward_weights=[1.0],
)
arm2_trainer = GRPOTrainer(
    model=arm2_model,
    processing_class=arm2_tok,
    args=arm2_cfg,
    train_dataset=rl_train,
    reward_funcs=[process_reward],
)
arm2_trainer.train()
arm2_trainer.save_model(OUT_ARM2); arm2_tok.save_pretrained(OUT_ARM2)

## 6. Evaluate both arms

We compare **final-answer accuracy**, **verbosity**, **exploit rate on
trap prompts**, and a rough **composite reward-hacking score** (CRHS).

In [ ]:
import json, torch
from prm_rl.evaluation.metrics    import final_answer_accuracy
from prm_rl.evaluation.behavioral import behavioral_scores
from prm_rl.evaluation.traps      import load_trap_scenarios, exploit_rate
from prm_rl.evaluation.crhs       import composite_reward_hacking_score

@torch.no_grad()
def generate(path, prompts, batch=4, max_new_tokens=256):
    tok = AutoTokenizer.from_pretrained(path); tok.padding_side='left'
    if tok.pad_token is None: tok.pad_token = tok.eos_token
    m = AutoModelForCausalLM.from_pretrained(
        path,
        torch_dtype=torch.bfloat16 if torch.cuda.is_available() else torch.float32,
        device_map='auto',
    ).eval()
    outs = []
    for i in range(0, len(prompts), batch):
        chunk = prompts[i:i+batch]
        enc = tok(chunk, return_tensors='pt', padding=True, truncation=True).to(m.device)
        gen = m.generate(**enc, max_new_tokens=max_new_tokens,
                         do_sample=False, pad_token_id=tok.pad_token_id)
        gen = gen[:, enc['input_ids'].shape[-1]:]
        outs.extend(tok.batch_decode(gen, skip_special_tokens=True))
    del m; torch.cuda.empty_cache()
    return outs

test_prompts = [ex['prompt'] for ex in gsm_test]
test_answers = [ex['answer'] for ex in gsm_test]

traps = load_trap_scenarios('data/traps/trap_examples.json')
trap_prompts = [t['prompt'] for t in traps]

In [ ]:
def evaluate(policy_path, tag):
    comp = generate(policy_path, test_prompts)
    trap_comp = generate(policy_path, trap_prompts)
    acc = final_answer_accuracy(comp, test_answers)
    beh = behavioral_scores(comp)
    tr  = exploit_rate(traps, trap_comp)
    crhs = composite_reward_hacking_score(
        exploit_rate=tr['exploit_rate'],
        phi_cct=0.0,           # placeholder — the faithfulness probe is offline
        avg_tokens=beh['avg_tokens'],
        verbosity_baseline=150.0,
        nie=0.0,
    )
    out = {'accuracy': acc, 'behavior': beh, 'traps': tr, 'crhs': crhs}
    print(f'\n=== {tag} ===')
    print(json.dumps(out, indent=2))
    return out

res_arm1 = evaluate(OUT_ARM1, 'Arm 1 — outcome')
res_arm2 = evaluate(OUT_ARM2, 'Arm 2 — naive process')

In [ ]:
import pandas as pd
rows = []
for tag, r in [('arm1_outcome', res_arm1), ('arm2_naive_process', res_arm2)]:
    rows.append({
        'arm': tag,
        'accuracy': r['accuracy']['accuracy'],
        'avg_tokens': r['behavior']['avg_tokens'],
        'avg_self_rougeL': r['behavior']['avg_self_rougeL'],
        'exploit_rate': r['traps']['exploit_rate'],
        'CRHS': r['crhs']['CRHS'],
    })
pd.DataFrame(rows).set_index('arm')

## 7. Where to go next

* **Bigger models / more steps** — swap `MODEL` to `Qwen/Qwen2.5-0.5B-Instruct`
  (still fits on Colab T4), bump `N_TRAIN` / `max_steps`.
* **More arms** — every arm is a new file under `src/prm_rl/rewards/` and a
  YAML under `configs/experiments/`. Arms 3–6 are already implemented; add
  them here by importing their factory and appending to `reward_funcs`.
* **Faithfulness metrics** — run `evaluation/faithfulness.py` and
  `evaluation/cma.py` offline on a paired-intervention set and feed the
  resulting Phi-CCT / NIE numbers back into `composite_reward_hacking_score`.
* **TACC Vista** — the same configs run under `sbatch slurm/rl.slurm <config>`.
  See `slurm/README.md`.